# Preprocessing and Feature Extraction

We build the feature table used for stress classification on the WESAD dataset. Only the
chest-worn RespiBAN signals are used here (ECG, EDA, EMG, respiration, temperature), sampled at 700 Hz.

Processing follows the order used in the paper: each signal is first filtered as a whole, then
segmented into overlapping 60-second windows (5-second stride) that never cross a condition
boundary, and finally summarised into features. Filtering the full signal before windowing
gives more reliable ECG R-peak detection than filtering isolated segments.

For every window we extract time-domain, frequency-domain and non-linear features, broadly
following Table 1 of Schmidt et al. (2018). Following Setz et al. (2010), the electrodermal
features are enriched with peak-distribution descriptors (amplitude spread and peak rate), and
every feature is additionally expressed relative to the subject's own baseline. Baseline is a
known condition by design, so this personal normalisation is computed per subject and does not
constitute leakage; the absolute and relative feature sets are both kept so the later selection
step can compare them.

This feature set is deliberately kept identical to the chest half of
`00_feature_extraction_combined.ipynb` (same signals, same formulas, same feature names once the
`c_` device tag used there is stripped), so the chest-only and combined tables stay comparable.

Global normalisation and feature selection are intentionally left out here: with only 15
subjects and a leave-one-subject-out evaluation, anything fitted from the data must stay inside
the cross-validation folds. This notebook stops at the raw feature table (`features.csv`),
keeping the `subject_id` column needed for the subject-wise split.


## Imports

In [ ]:
import os
import pickle # WESAD ships each subject as a pickled dict (signals + labels)
import warnings

import numpy as np
import pandas as pd
import neurokit2 as nk # signal cleaning and peak/cycle detection (ECG, EDA, EMG, respiration)
from scipy.signal import welch # power spectral density, for the EDA/respiration band-power features
from scipy.integrate import trapezoid # integrates the PSD over each frequency band


warnings.filterwarnings("ignore") # NeuroKit2 emits per-window edge-case warnings across 6k+ windows; suppressed for readability

## Configuration

`DATA_DIR` points to the folder containing the per-subject directories `S2`, `S3`, ..., `S17`.
Subjects S1 and S12 are excluded, as in the original dataset.


In [ ]:
DATA_DIR = os.path.join(os.getcwd(), "..", "..", "WESAD")   # WESAD folder at the repository root
OUT_DIR = "chest_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

SUBJECTS = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]

# WESAD's raw label field spans 0-7 (0 = undefined/transient, 5-7 = unused conditions); 
# only the three studied conditions are kept
FS = 700                                       # chest sampling rate (Hz)
LABEL_MAP = {1: "baseline", 2: "stress", 3: "amusement"}
KEEP_LABELS = set(LABEL_MAP)

WINDOW_SEC, STRIDE_SEC = 60, 5
WINDOW_LEN = WINDOW_SEC * FS
STRIDE_LEN = STRIDE_SEC * FS

# Plausible range for skin temperature; values outside indicate sensor detachment
TEMP_MIN, TEMP_MAX = 20.0, 45.0

# Plausible range for breathing rate (breaths/min): below this is apnea-like and above it is
# extreme tachypnea, so values outside are treated as a respiration-detection artefact rather
# than a real physiological rate (see the RESP plausibility check below)
RESP_RATE_MIN, RESP_RATE_MAX = 4.0, 40.0

## Loading and segmentation

In [ ]:
def load_subject(subject_id):
    """Return the five chest signals and the label vector for one subject."""
    path = os.path.join(DATA_DIR, f"S{subject_id}", f"S{subject_id}.pkl")
    with open(path, "rb") as f:
        data = pickle.load(f, encoding="latin1")
    chest = data["signal"]["chest"]
    return {
        "ECG": np.asarray(chest["ECG"], dtype=float).ravel(),
        "EDA": np.asarray(chest["EDA"], dtype=float).ravel(),
        "EMG": np.asarray(chest["EMG"], dtype=float).ravel(),
        "Resp": np.asarray(chest["Resp"], dtype=float).ravel(),
        "Temp": np.asarray(chest["Temp"], dtype=float).ravel(),
        "label": np.asarray(data["label"], dtype=int).ravel(),  # per-sample condition code, 700 Hz
    
    }


def find_segments(labels):
    """Return (label, start, end) for each contiguous run of a kept condition."""
    segments = []
    i, n = 0, len(labels)
    while i < n:
        if labels[i] in KEEP_LABELS:
            j = i
            while j < n and labels[j] == labels[i]:
                j += 1
            segments.append((int(labels[i]), i, j))
            i = j
        else:
            i += 1
    return segments


def clean_temperature(temp):
    """Replace out-of-range temperature samples (sensor dropout) by interpolation."""
    t = temp.copy()
    t[(t < TEMP_MIN) | (t > TEMP_MAX)] = np.nan
    t = pd.Series(t).interpolate(limit_direction="both").to_numpy()
    return np.nan_to_num(t, nan=np.nanmedian(temp)) # safety net, in case any NaN survives interpolation

## Non-linear helpers

Sample entropy and the Poincaré descriptors are computed on the RR-interval series (a few
dozen values per window), which is inexpensive.


In [ ]:
def sample_entropy(series, m=2, r_factor=0.2):
    """Sample entropy of a short series (e.g. RR intervals)."""
    x = np.asarray(series, dtype=float)
    n = len(x)
    if n < m + 2:
        return np.nan
    r = r_factor * np.std(x) # tolerance, scaled to the series' own variability
    if r == 0:
        return np.nan

    def count_matches(dim):
        templates = np.array([x[i:i + dim] for i in range(n - dim + 1)])
        total = 0
        for i in range(len(templates)):
            dist = np.max(np.abs(templates - templates[i]), axis=1) # Chebyshev distance to every other template
            total += np.count_nonzero(dist <= r) - 1     # exclude self-match
        return total

    b = count_matches(m)
    a = count_matches(m + 1)
    if a == 0 or b == 0:
        return np.nan
    return -np.log(a / b)


def poincare_sd(rr):
    """Poincaré plot descriptors SD1, SD2 from RR intervals (ms)."""
    if len(rr) < 3:
        return np.nan, np.nan
    sdsd = np.std(np.diff(rr), ddof=1) # short-term variability (successive differences)
    sdnn = np.std(rr, ddof=1)          # overall variability
    sd1 = np.sqrt(0.5) * sdsd          # short-term (beat-to-beat) variability
    var2 = 2 * sdnn ** 2 - 0.5 * sdsd ** 2
    sd2 = np.sqrt(var2) if var2 > 0 else np.nan # long-term variability
    return sd1, sd2 


def band_power(signal, fs, low, high):
    """Total power and peak frequency of `signal` within [low, high] Hz (Welch)."""
    nperseg = min(len(signal), fs * 30)
    freqs, psd = welch(signal, fs=fs, nperseg=nperseg)
    band = (freqs >= low) & (freqs <= high)
    if not np.any(band):
        return np.nan, np.nan
    power = trapezoid(psd[band], freqs[band])       # integrate PSD over the band
    peak_freq = freqs[band][np.argmax(psd[band])]   # frequency of maximum spectral power in the band
    return power, peak_freq

## Feature extraction

For each window the following features are computed:

- **ECG / HRV** — mean and std heart rate; mean NN interval, SDNN, RMSSD, pNN50; LF/HF ratio;
  non-linear Poincaré SD1/SD2 and sample entropy of the RR series. All ECG/HRV features are
  computed with the same formulas as the chest branch of `00_feature_extraction_combined.ipynb`.
- **EDA** — tonic level (SCL) and clean-signal statistics; number, amplitude spread, rate and
  inter-peak interval of SCR peaks (Setz-style peak distribution); spectral power of the signal.
- **EMG** — mean absolute value, RMS and waveform length of the cleaned signal, and the number of
  detected muscle activations (few features, as EMG mainly serves as a movement-artefact check).
- **Respiration** — mean/std breathing rate, amplitude, inhalation/exhalation ratio, and the
  dominant respiratory frequency and its power.
- **Temperature** — mean, std, min, max, range, slope.


In [ ]:
def window_features(ecg, eda, rsp, temp, emg_clean=None, emg_onsets=None):
    """Feature dictionary for a single 60-second window."""
    feats = {}

    # ECG / heart-rate variability
    r_peaks = np.flatnonzero(ecg["ECG_R_Peaks"].to_numpy() == 1)
    if len(r_peaks) >= 4:                               
        rr = np.diff(r_peaks) / FS * 1000.0            # RR intervals (ms)
        diff_rr = np.diff(rr)

        hr = 60000.0 / rr                               # beat-to-beat heart rate (bpm)
        feats["HR_mean"], feats["HR_std"] = np.mean(hr), np.std(hr)

        feats["HRV_MeanNN"] = np.mean(rr)
        feats["HRV_SDNN"] = np.std(rr, ddof=1)
        feats["HRV_RMSSD"] = np.sqrt(np.mean(diff_rr ** 2))
        feats["HRV_pNN50"] = np.mean(np.abs(diff_rr) > 50) * 100.0
        sd1, sd2 = poincare_sd(rr)
        feats["HRV_SD1"], feats["HRV_SD2"] = sd1, sd2
        feats["HRV_SampEn"] = sample_entropy(rr) if len(rr) >= 10 else np.nan

        try:
            # LF/HF from peak times rebuilt from the RR series (same reconstruction used in the
            # combined notebook), instead of from the raw R-peak sample indices
            ps = np.round(np.cumsum(np.concatenate([[0.0], rr])) / 1000.0 * FS).astype(int)
            lfhf = nk.hrv_frequency({"ECG_R_Peaks": ps}, sampling_rate=FS, show=False)
            feats["HRV_LFHF"] = float(lfhf["HRV_LFHF"].iloc[0])
        except Exception:
            feats["HRV_LFHF"] = np.nan
    else:
        for k in ("HR_mean", "HR_std", "HRV_MeanNN", "HRV_SDNN", "HRV_RMSSD", "HRV_pNN50",
                  "HRV_SD1", "HRV_SD2", "HRV_SampEn", "HRV_LFHF"):
            feats[k] = np.nan

    # Electrodermal activity
    clean = eda["EDA_Clean"].to_numpy()
    tonic = eda["EDA_Tonic"].to_numpy()
    phasic = eda["EDA_Phasic"].to_numpy()
    feats["SCL_mean"] = np.mean(tonic)
    feats["SCL_std"] = np.std(tonic)
    feats["EDA_mean"] = np.mean(clean)
    feats["EDA_std"] = np.std(clean)
    feats["EDA_min"] = np.min(clean)
    feats["EDA_max"] = np.max(clean)
    feats["EDA_range"] = feats["EDA_max"] - feats["EDA_min"]
    feats["EDA_slope"] = np.polyfit(np.arange(len(clean)), clean, 1)[0]

    scr_peaks = np.flatnonzero(eda["SCR_Peaks"].to_numpy() == 1)
    heights = phasic[scr_peaks]
    feats["SCR_count"] = int(len(scr_peaks))
    feats["SCR_amp_mean"] = float(np.mean(heights)) if len(scr_peaks) else 0.0
    feats["SCR_amp_std"] = float(np.std(heights)) if len(scr_peaks) > 1 else 0.0
    feats["SCR_rate"] = len(scr_peaks) / WINDOW_SEC * 60.0           # peaks per minute
    if len(scr_peaks) >= 2:
        ipi = np.diff(scr_peaks) / FS                               # inter-peak interval (s)
        feats["SCR_ipi_mean"] = float(np.mean(ipi))
    else:
        feats["SCR_ipi_mean"] = np.nan
    feats["EDA_psd_power"], _ = band_power(clean, FS, 0.05, 0.5)

    # EMG (few features, from the cleaned signal)
    if emg_clean is not None:
        feats["EMG_mav"] = np.mean(np.abs(emg_clean))                  # mean absolute value
        feats["EMG_rms"] = np.sqrt(np.mean(emg_clean ** 2))            # root mean square
        feats["EMG_wl"] = np.sum(np.abs(np.diff(emg_clean)))           # waveform length
        feats["EMG_activations"] = (int(np.nansum(emg_onsets == 1))
                                     if emg_onsets is not None else np.nan)
    else:
        feats["EMG_mav"] = feats["EMG_rms"] = feats["EMG_wl"] = feats["EMG_activations"] = np.nan

    # Respiration
    feats["RESP_rate_mean"] = np.nanmean(rsp["RSP_Rate"].to_numpy())
    feats["RESP_rate_std"] = np.nanstd(rsp["RSP_Rate"].to_numpy())
    feats["RESP_amp_mean"] = np.nanmean(rsp["RSP_Amplitude"].to_numpy())
    phase = rsp["RSP_Phase"].to_numpy()
    insp, exp = np.count_nonzero(phase == 1), np.count_nonzero(phase == 0)
    feats["RESP_IE_ratio"] = insp / exp if exp > 0 else np.nan
    power, peak = band_power(rsp["RSP_Clean"].to_numpy(), FS, 0.1, 0.5)
    feats["RESP_psd_power"], feats["RESP_peak_freq"] = power, peak

    # Temperature
    feats["TEMP_mean"] = np.mean(temp)
    feats["TEMP_std"] = np.std(temp)
    feats["TEMP_min"] = np.min(temp)
    feats["TEMP_max"] = np.max(temp)
    feats["TEMP_range"] = feats["TEMP_max"] - feats["TEMP_min"]
    feats["TEMP_slope"] = np.polyfit(np.arange(len(temp)), temp, 1)[0]

    return feats


def process_subject(subject_id):
    """Filter the whole signal of one subject, then extract features per window."""
    s = load_subject(subject_id)
    ecg, _ = nk.ecg_process(s["ECG"], sampling_rate=FS)
    eda, _ = nk.eda_process(s["EDA"], sampling_rate=FS)
    rsp, _ = nk.rsp_process(s["Resp"], sampling_rate=FS)
    try:
        emg, _ = nk.emg_process(s["EMG"], sampling_rate=FS)
        emg_clean = emg["EMG_Clean"].to_numpy()
        emg_onsets = emg["EMG_Onsets"].to_numpy()
    except Exception:
        emg_clean = emg_onsets = None
    temp = clean_temperature(s["Temp"])
    
# Windows are generated per contiguous segment, never across a condition boundary
    rows = []
    for label, seg_start, seg_end in find_segments(s["label"]):
        start = seg_start
        while start + WINDOW_LEN <= seg_end:
            window = slice(start, start + WINDOW_LEN)
            feats = window_features(ecg.iloc[window], eda.iloc[window],
                                    rsp.iloc[window], temp[window],
                                    emg_clean[window] if emg_clean is not None else None,
                                    emg_onsets[window] if emg_onsets is not None else None)
            feats.update(label=label, label_name=LABEL_MAP[label], subject_id=subject_id)
            rows.append(feats)
            start += STRIDE_LEN
    return rows

## Building the feature table

Each subject is processed in turn. This is the expensive step, as the full-length signals are
filtered before segmentation.


In [ ]:
rows = []
for sid in SUBJECTS:
    subject_rows = process_subject(sid)
    rows.extend(subject_rows)
    print(f"S{sid}: {len(subject_rows)} windows")

features = pd.DataFrame(rows)
# Separate identifying/label columns from the actual predictors, so downstream code can
# select just the feature_cols without hardcoding names
meta_cols = ["label", "label_name", "subject_id"]
feature_cols = [c for c in features.columns if c not in meta_cols]
print(f"\n{features.shape[0]} windows, {len(feature_cols)} base features")

S2: 390 windows
S3: 398 windows
S4: 399 windows
S5: 409 windows
S6: 407 windows
S7: 406 windows
S8: 407 windows
S9: 406 windows
S10: 422 windows
S11: 412 windows
S13: 411 windows
S14: 412 windows
S15: 413 windows
S16: 410 windows
S17: 421 windows

6123 windows, 40 base features


## Baseline-relative features (Setz et al., 2010)

Each feature is additionally expressed relative to the subject's own baseline mean. Baseline is
a known condition, so this per-subject normalisation is legitimate. Both the absolute and the
`_rel` versions are kept, letting the selection step compare relative vs. non-relative features.


In [ ]:
rel_parts = []
for _, group in features.groupby("subject_id", sort=False):
    # Mean of each feature over this subject's own baseline windows only
    baseline_mean = group.loc[group["label_name"] == "baseline", feature_cols].mean()
    # Mean-centered on the subject's baseline (not z-scored): baseline windows end up ~0,
    # stress/amusement windows show the deviation from that subject's own resting level
    rel_parts.append(group[feature_cols] - baseline_mean)

relative = pd.concat(rel_parts).sort_index()        # restore original row order after groupby
relative.columns = [f"{c}_rel" for c in feature_cols]
features = pd.concat([features, relative], axis=1)
print(f"{features.shape[0]} windows, {features.shape[1]} columns "
      f"({len(feature_cols)} absolute + {len(feature_cols)} relative + {len(meta_cols)} meta)")

6123 windows, 83 columns (40 absolute + 40 relative + 3 meta)


## Quality check

In addition to the missingness check, `RESP_rate_mean` is screened against a plausible
physiological range (see `RESP_RATE_MIN`/`RESP_RATE_MAX` above). Unlike EDA/ECG, `neurokit2`'s
respiration pipeline has no built-in artefact rejection, so a bad peak/trough detection in
`nk.rsp_process` can silently produce an implausible rate (e.g. close to 0 breaths/min). This
check does not drop or alter any window — it only flags them for inspection, since a handful of
outlier windows do not materially affect downstream modelling.


In [8]:
print(features["label_name"].value_counts(), "\n")

missing = features.isna().sum()
print("Columns with missing values:")
print(missing[missing > 0] if missing.any() else "none")

implausible_resp = (features["RESP_rate_mean"] < RESP_RATE_MIN) | (features["RESP_rate_mean"] > RESP_RATE_MAX)
n_bad = int(implausible_resp.sum())
print(f"\nWindows with implausible RESP_rate_mean (outside {RESP_RATE_MIN:.0f}-{RESP_RATE_MAX:.0f} "
      f"breaths/min): {n_bad} ({n_bad / len(features) * 100:.1f}%)")
if n_bad:
    print(features.loc[implausible_resp, ["subject_id", "label_name", "RESP_rate_mean"]]
          .sort_values("RESP_rate_mean").to_string(index=False))

label_name
baseline     3354
stress       1826
amusement     943
Name: count, dtype: int64 

Columns with missing values:
HRV_SampEn             48
SCR_ipi_mean         4653
RESP_IE_ratio           5
HRV_SampEn_rel         48
SCR_ipi_mean_rel     5415
RESP_IE_ratio_rel       5
dtype: int64

Windows with implausible RESP_rate_mean (outside 4-40 breaths/min): 42 (0.7%)
 subject_id label_name  RESP_rate_mean
          8   baseline        0.857883
          8   baseline        0.871604
          8   baseline        0.873249
          8   baseline        0.913993
          8   baseline        0.959774
          8   baseline        0.981781
          8   baseline        0.992227
          8   baseline        1.004186
          8   baseline        1.012944
          8   baseline        1.046686
          8   baseline        1.082494
          8   baseline        1.086555
          8   baseline        1.090596
          8   baseline        1.132985
          8   baseline        1.187900
      

## Saving

In [9]:
features.to_csv(os.path.join(OUT_DIR, "features.csv"), index=False)
print(f"Saved {features.shape[0]} rows to {os.path.join(OUT_DIR, 'features.csv')}")

Saved 6123 rows to outputs\features.csv
